# Cardioformer-CKD — End-to-end experiments

Prognostic incident-CKD prediction from **MIMIC-IV + MIMIC-IV-ECG**, adapting the
Cardioformer ECG transformer. Run cells top to bottom. Requires credentialed
PhysioNet data (no data is shipped in this repo).


## 0. Setup


In [ ]:
# !pip install -r requirements.txt
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
MIMIC = '/path/to/mimiciv/3.1'        # contains hosp/
ECG   = '/path/to/mimic-iv-ecg/1.0'   # contains record_list.csv + files/
HOSP  = f'{MIMIC}/hosp'
OUT   = 'dataset/ckd'


## 1. Build the incident-CKD cohort (index times, exclusions, labels, censoring)

Default horizon = 2 years. Change `--horizon_days` to 365 (1y) or 1825 (5y).

In [ ]:
!python -m data_preprocessing.build_cohort \
  --patients   {HOSP}/patients.csv.gz \
  --admissions {HOSP}/admissions.csv.gz \
  --diagnoses  {HOSP}/diagnoses_icd.csv.gz \
  --labevents  {HOSP}/labevents.csv.gz \
  --ecg_record_list {ECG}/record_list.csv \
  --out_dir {OUT} --horizon_days 730 --blank_days 30 \
  --baseline_lookback_days 365 --n_intervals 8

## 2. Extract structured EHR features (labs / vitals / comorbidities before t0)

In [ ]:
!python -m data_preprocessing.extract_ehr_features \
  --cohort {OUT}/ckd_cohort_labels.parquet \
  --patients {HOSP}/patients.csv.gz --admissions {HOSP}/admissions.csv.gz \
  --diagnoses {HOSP}/diagnoses_icd.csv.gz --labevents {HOSP}/labevents.csv.gz \
  --out_dir {OUT}

## 3. Preprocess ECG waveforms (WFDB -> filtered, resampled, normalized .npy)

In [ ]:
!python -m data_preprocessing.extract_ecg \
  --cohort {OUT}/ckd_cohort_labels.parquet \
  --ecg_root {ECG} --out_dir {OUT}/ecg_npy --target_fs 250

## 4. Link modalities and make subject-independent splits

In [ ]:
!python -m data_preprocessing.link_ecg_ehr \
  --cohort {OUT}/ckd_cohort_labels.parquet --ehr {OUT}/ehr_features.parquet \
  --ecg_manifest {OUT}/ecg_npy/ecg_manifest.parquet --out_dir {OUT} --policy all
!python -m data_preprocessing.make_splits \
  --dataset {OUT}/ckd_dataset.parquet --out_dir {OUT} \
  --train_frac 0.6 --val_frac 0.2 --seed 41

## 5. Quick label-balance / cohort sanity check

In [ ]:
import pandas as pd
df = pd.read_parquet(f'{OUT}/ckd_dataset_split.parquet')
print(df.groupby('split')['label_binary'].agg(['count','mean']))
print('event rate (any):', df['event_indicator'].mean())

## 6. Train — multimodal, discrete-time survival head (default)

In [ ]:
!python run.py --config configs/ckd_prognosis.yaml \
  --setting ckd_2y_survival_multimodal --is_training 1

## 7. Ablations

Isolate each modality and compare the survival vs binary head.

In [ ]:
# ECG only
!python run.py --config configs/ckd_prognosis.yaml \
  --setting ckd_2y_ecg_only --use_ecg 1 --use_ehr 0 --is_training 1
# EHR only
!python run.py --config configs/ckd_prognosis.yaml \
  --setting ckd_2y_ehr_only --use_ecg 0 --use_ehr 1 --is_training 1
# Fixed-horizon binary head
!python run.py --config configs/ckd_prognosis.yaml \
  --setting ckd_2y_binary --head binary --is_training 1

## 8. Inspect predicted cumulative-incidence curves (survival head)

In [ ]:
import torch, json
from types import SimpleNamespace
from models.CardioformerCKD import CardioformerCKD
# Load args used for the run (edit the setting as needed), rebuild model, load ckpt,
# then: surv, cif = model.predict_cif(x_ecg=..., ehr=...)
print('See exp/exp_ckd_prognosis.py validate() for the full eval loop.')

## 9. Collect test metrics

In [ ]:
import json, glob
for f in glob.glob('results/**/test_metrics.json', recursive=True):
    print(f, json.load(open(f)))